## Example code for using pandas for the python 5 (week 10) programming task

The code below doesnt produce the same output. Instead, it will introduce `pandas` and will use dataframe operations to concatenate, by column the data from each data logger file. Remember, each file essentially has four columns: index, date, time, and temperature. We want only date, time, AM/PM, and temperature, and we want the heading for each column to be some representation of the file name. 

We will then use the symmetrical data frame to explore pandas a bit as an alternative tutorial to what we will cover next week in class.

#### Import libraries

Here we are using four libraries. What each library does:

- `glob`: Searches for files and directories using patterns (wildcards). For example, glob.glob("*.txt") finds all text files in the current directory. This is extremely useful when you want to loop through many files automatically instead of typing their names one by one.

- `os`:Provides a way to interact with your operating system (for example, renaming files, joining paths, or getting file names from full paths). We’ll use it later to clean file names and extract only the base names for labeling our data.

- `io`: Allows Python to treat text strings as if they were files. This is handy when you’ve read raw text into memory and want to process it as a file-like object using commands such as `pd.read_csv()`.

- `pandas` — A data analysis library that provides the DataFrame object, which makes it easy to read, clean, combine, and analyze tabular data efficiently. If you don’t already have pandas installed, you can install it from the command line:
```sh

pip install pandas # using pip
conda install pandas # if you are using anaconda
```

In [1]:
import glob, os, io
import pandas as pd

#### Read files using `glob`
- Code below will store all filenames in directory ending in `.txt.txt` into `filelist`
- for loop below is just printing elements of the list to confirm this is working as expected. The code here is just a demo so that you can clearly see how `glob` works.


In [2]:
filelist = glob.glob('*.txt.txt')
for file in filelist:
    print(file)

1901302225_H12_D_13.txt.txt
1901302194_H10_S_14.txt.txt
1901302237_H10_D_13.txt.txt
1901302157_H12_S_13.txt.txt
1901302225_H12_D_14.txt.txt
1901302194_H10_S_13.txt.txt
1901302237_H10_D_14.txt.txt
1901302157_H12_S_14.txt.txt


### Defining a function to safely read each tab-delimited file

We haven't seen functions yet in python, so here you go. This chunk of code defines a function that can be called later as `read_one_tab_file()`, where the filename or path will be fed as an argument in parentheses. This will later be used instead of `open` or `pd.read`.

Many data loggers (and even sequencing or instrument software) export text files that can contain hidden **NULL characters** (`\x00`) or other odd encodings. These can cause `UnicodeDecodeError` or prevent `pandas` or `open` from reading the file correctly. The function below — `read_one_tab_file()` — handles those issues and ensures each file is read cleanly into a pandas DataFrame.

#### What this function does

- **Opens the file in binary mode** (`"rb"`) so we can clean low-level issues before decoding.  
- **Replaces NULL characters** (`\x00`) with nothing — these are common “gremlins” from sensor or instrument data exports.  
- **Decodes the byte stream** — first tries UTF-8 (most common), and if that fails, falls back to Latin-1 (which safely includes special characters like ° for temperature).  
- **Reads the cleaned text into a pandas DataFrame**, using tab (`\t`) as the separator.  
- **Strips whitespace** from column headers, ensuring they match exactly when we refer to them later.


In [3]:
def read_one_tab_file(path):
    # Strip NULLs and decode (latin-1 safely handles the ° symbol)
    with open(path, "rb") as f:
        raw = f.read().replace(b"\x00", b"")
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")

    # Logger exports are TAB-separated
    df = pd.read_csv(io.StringIO(text), sep="\t")
    df.columns = [c.strip() for c in df.columns]  # tidy headers
    return df

### Listing and confirming input files with `glob`

Here we are using `glob` to automatically find all of the logger data files.

The code below looks through the current working directory for all files ending with `.txt.txt`, and then stores the filenames in `filelist`.  `print` statement here is just used to confirm and illustrate that this is working.

In [12]:
filelist = glob.glob("*.txt.txt")
print("Found", len(filelist), "files")


Found 8 files


### Initializing `blocks` list for use later.

Below we are initializing the list `blocks` so that we can grow it later. Each time the below loop runs, we’ll append a new dataframe (representing one file’s data) to this list before using`pandas` functions such as `pd.concat()` or ``pd.merge()` to combine everything in `blocks` into one structured dataframe.

In [14]:
blocks = []

### Looping through all files and building clean DataFrames

Using the `read_one_tab_file()`, we can loop through all of the text files that we found with `glob` to:

- Read each file using our cleaning function.
- Create a timestamp column from the file’s `Date` and `Time` fields.
- Extract temperature readings and AM/PM information.
- Name each column set based on the file from which data in the column originated.
- Append each resulting DataFrame to the `blocks` list.

#### Step-by-step explanation

- **Reading each file**  
  The function `read_one_tab_file(fn)` opens and cleans one tab-delimited file at a time, returning a pandas DataFrame.  
  This keeps our code modular and allows us to reuse the same logic later if we want to process other similar files.

- **Building a timestamp**  
  The code combines the `Date` and `Time` columns into a single string (e.g., `"2012-09-30 08:00 AM"`) and converts it into a datetime object using `pd.to_datetime()`.  
  Using `errors="coerce"` means that if any rows have bad formatting, they’ll be converted to `NaT` (Not-a-Time) instead of causing the entire script to fail.

- **Extracting temperature**  
  The column `'Readings (°F)'` is converted to numeric values with `pd.to_numeric()`.  
  This function safely converts text to numbers and turns any non-numeric entries (e.g., missing or corrupted data) into `NaN`, which pandas can handle gracefully.

- **Extracting AM/PM**  
  The AM/PM label is stored inside the time field, so we use a **regular expression** to find whether each entry includes "AM" or "PM".  
  The pattern `r'(?i)\b(AM|PM)\b'` ignores case and only matches standalone AM or PM strings (not parts of other words).  
  We then convert all results to uppercase for consistency.

- **Naming by file**  
  The `stem` variable isolates just the base of the filename (everything before `.txt.txt`).  
  This ensures each file contributes a unique pair of columns like `H18_S_14_F` and `H18_S_14_AMPM`, letting us trace data back to its source file.

- **Creating a per-file DataFrame**  
  We construct a small DataFrame for each file with timestamp as the index and temperature and AM/PM as columns.  
  Using `.values` prevents pandas from trying to align indexes automatically across columns of different lengths, which could otherwise insert unwanted `NaN`s.

- **Appending to `blocks`**  
  Each cleaned DataFrame (`block`) is added to our growing list `blocks`.  
  By the end of the loop, `blocks` will contain one DataFrame per file — all neatly formatted and ready to merge together into a unified dataset.


In [15]:
for fn in filelist:
    df = read_one_tab_file(fn)

    # Timestamp from Date + Time (AM/PM is handled by to_datetime)
    date_str = df["Date"].astype(str).str.strip()
    time_str = df["Time"].astype(str).str.strip()
    ts = pd.to_datetime(date_str + " " + time_str, errors="coerce")

    # (Optional but helpful) round to the nearest minute to improve alignment
    ts = ts.dt.round("T")

    # Temperature column (adjust name if your header differs)
    tempF = pd.to_numeric(df["Readings (°F)"], errors="coerce")

    # Column name from filename (strip the .txt.txt)
    stem = os.path.splitext(os.path.splitext(os.path.basename(fn))[0])[0]

    # IMPORTANT: use .values so pandas doesn't try to align by row index
    block = pd.DataFrame(
        {f"{stem}_F": tempF.values},
        index=ts
    )
    blocks.append(block)

/var/folders/m2/04t_0bnx1rv6940_lc07qdbm0000gn/T/ipykernel_44813/2253128440.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(date_str + " " + time_str, errors="coerce")


### Combining all DataFrames into one dataframe

Now that we have one small dataframe per input file stored in our list `blocks`, we can merge them all together into a single large dataframe.  
This is done with the `pandas.concat()` function, which can combine multiple dataframes either **row-wise** (`axis=0`) or **column-wise** (`axis=1`).

Here we use `axis=1` to align the data **by timestamp** and bind the columns side by side — similar to `cbind()` in R.

Since each of the smaller dataframes in `block` share the same timestamp index, `pd.concat()` automatically lines up data from the same date and time across all sensors or files. Where timestamps are missing from some files, pandas will fill those cells with NaN, preserving the time alignment.


The resulting variable merged is now one large, well-structured DataFrame that combines all of the temperature and AM/PM columns from every file in the directory.


In [16]:
merged = pd.concat(blocks, axis=1)

### Sorting the merged DataFrame by timestamp

After merging dataframes, it’s good to make sure the rows are in chronological order. The code below checks whether the `pandas` dataframe’s index is actually a datetime object and, if so, sorts it from earliest to latest.

In [18]:
# Sort by time if the index is datetime
if pd.api.types.is_datetime64_any_dtype(merged.index):
    merged = merged.sort_index()
merged.shape

(42713, 8)

#### optional, drop rows that consist of all NaN

In [19]:
merged = merged.dropna(how="all")
merged.shape

(42713, 8)

### Writing the combined dataframe to a .csv file

The final step exports cleaned and merged dataset so it can be easily shared or analyzed later.  
The code below writes the fataframe to a file, making sure that the timestamps are clearly labeled as the first column.



In [20]:
# Give the index a name so the first CSV cell isn't blank
merged.to_csv("cbind_by_timestamp.csv", index_label="timestamp")
print("Wrote cbind_by_timestamp.csv with shape:", merged.shape)
merged.head()

Wrote cbind_by_timestamp.csv with shape: (42713, 8)


,1901302225_H12_D_13_F,1901302194_H10_S_14_F,1901302237_H10_D_13_F,1901302157_H12_S_13_F,1901302225_H12_D_14_F,1901302194_H10_S_13_F,1901302237_H10_D_14_F,1901302157_H12_S_14_F
2012-09-30 08:00:00,41.4,NaN,45.9,42.4,NaN,44.1,NaN,NaN
2012-09-30 08:35:00,41.6,NaN,46.0,42.6,NaN,43.9,NaN,NaN
2012-09-30 09:10:00,42.0,NaN,46.2,42.9,NaN,44.1,NaN,NaN
2012-09-30 09:45:00,42.3,NaN,46.6,43.2,NaN,44.2,NaN,NaN
2012-09-30 10:20:00,42.7,NaN,47.3,43.7,NaN,44.4,NaN,NaN
